In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType

In [ ]:
# SPARK SESSION
spark = SparkSession.builder \
    .appName("Analisis_Clientes_Behavioural") \
    .getOrCreate()

In [ ]:
df_beh = spark.read.parquet("/home/jovyan/work/data/BEHAVIOURAL_CLEAN")
df_cli = spark.read.parquet("/home/jovyan/work/data/CLIENTS_CLEAN")

In [ ]:
# Intersección por la columna CLIENTS_ID
ids_comunes = df_beh.select("CLIENT_ID") \
                    .intersect(df_cli.select("CLIENT_ID"))

# Contar cuántos son iguales
count_comunes = ids_comunes.count()

print("Número de CLIENT_ID iguales:", count_comunes)

In [ ]:
df_comunes = df_beh.join(df_cli, on="CLIENT_ID", how="inner")

df_comunes.repartition(1) \
    .write.mode("overwrite") \
    .option("header", "true") \
    .csv("/home/jovyan/work/data/CLIENTES_COMUNES")

In [ ]:
# IDs únicos de beh que NO están en cli
ids_solo_beh = (
    df_beh.select("CLIENT_ID").distinct()
          .join(df_cli.select("CLIENT_ID").distinct(), "CLIENT_ID", "left_anti")
)

# Traer TODAS las columnas de df_beh de esos IDs únicos
solo_beh = df_beh.join(ids_solo_beh, "CLIENT_ID", "inner").dropDuplicates(["CLIENT_ID"])

solo_beh_count = solo_beh.count()

print("CLIENT_ID únicos solo en df_beh:", solo_beh_count)
solo_beh.show(solo_beh_count, truncate=False)

# IDs únicos de cli que NO están en beh
ids_solo_cli = (
    df_cli.select("CLIENT_ID").distinct()
          .join(df_beh.select("CLIENT_ID").distinct(), "CLIENT_ID", "left_anti")
)

# Traer TODAS las columnas de df_cli de esos IDs únicos
solo_cli = df_cli.join(ids_solo_cli, "CLIENT_ID", "inner").dropDuplicates(["CLIENT_ID"])

solo_cli_count = solo_cli.count()

print("CLIENT_ID únicos solo en df_cli:", solo_cli_count)
solo_cli.show(solo_cli_count, truncate=False)

In [ ]:
solo_beh.toPandas().to_csv("/home/jovyan/work/data/solo_beh.csv", index=False)
solo_cli.toPandas().to_csv("/home/jovyan/work/data/solo_cli.csv", index=False)

print("Archivos guardados en /home/jovyan/work/data/")

In [ ]:
solo_beh.show()

In [ ]:
solo_cli.show()

In [ ]:
# # 1) JOIN con todos los CLIENT_ID comunes y todas las columnas
# df_comunes = df_beh.join(df_cli, on="CLIENT_ID", how="inner")

# # 2) Spark escribe en una carpeta temporal
# df_comunes.repartition(1) \
#     .write.mode("overwrite") \
#     .option("header", "true") \
#     .csv("/home/jovyan/work/data/df_comunes_tmp")

# # 3) Mover el part-*.csv a un único df_comunes.csv en data/
# import os, shutil

# for f in os.listdir("/home/jovyan/work/data/df_comunes_tmp"):
#     if f.startswith("part-") and f.endswith(".csv"):
#         shutil.move(
#             "/home/jovyan/work/data/df_comunes_tmp/" + f,
#             "/home/jovyan/work/data/df_comunes.csv"
#         )
#         break

# shutil.rmtree("/home/jovyan/work/data/df_comunes_tmp")

# print("Creado /home/jovyan/work/data/df_comunes.csv")
